In [1]:
import json
import requests
import xml.etree.ElementTree as ET

from pathlib import Path
from datetime import datetime


import duckdb
import polyline
import pandas as pd
from geopy.distance import geodesic


from to_gpx import to_gpx

In [2]:
DATA_BASE_PATH = Path("./data").resolve().absolute()
GRAPHHOPPER_BASE_URL = "http://localhost:8989"
GPS_ACCURACY = 50

In [3]:
gps_data_path = DATA_BASE_PATH / "gps_data.parquet"
ground_truth_path = DATA_BASE_PATH / "ground_truth_route.parquet"
newson_krumm_route_network_path = DATA_BASE_PATH / "road_network.parquet"

In [4]:
route_network = duckdb.query(
    f"""
    SELECT *
    FROM '{newson_krumm_route_network_path}'
    """
).to_df()
route_network

,edge_id,from_node_id,to_node_id,two_way,speed,vertex_count,linestring
0,883991900000,883991900000,883991900001,1,22.222222,10,"LINESTRING(-122.732318937778 47.8899192810059,..."
1,883991900001,883991900002,883991900003,1,11.111111,4,"LINESTRING(-122.71107852459 47.8776508569717, ..."
2,883991900002,883991900004,883991900005,1,11.111111,29,"LINESTRING(-122.707419991493 47.8761515021324,..."
3,883991900003,883991900004,883991900002,1,11.111111,4,"LINESTRING(-122.707419991493 47.8761515021324,..."
4,883991900004,883991900003,883991900006,1,11.111111,23,"LINESTRING(-122.715329825878 47.8818699717522,..."
...,...,...,...,...,...,...,...
158162,884152400184,884152400122,884152400187,1,11.111111,3,"LINESTRING(-121.777908504009 47.4525502324104,..."
158163,884152400185,884152400183,884152400188,1,11.111111,7,"LINESTRING(-121.781320273876 47.4532207846642,..."
158164,884152400186,884152400184,884152400189,1,11.111111,7,"LINESTRING(-121.784308254719 47.4577805399895,..."
158165,884152400187,884152400190,884152400186,1,11.111111,2,"LINESTRING(-121.782489717007 47.4503803253174,..."


In [5]:
route_network.columns

Index(['edge_id', 'from_node_id', 'to_node_id', 'two_way', 'speed',
       'vertex_count', 'linestring'],
      dtype='object')

In [6]:
route_network["edge_id"].astype(str).apply(lambda x: x[:3]).unique()

array(['883', '884'], dtype=object)

In [7]:
def convert_edge_id_to_int32(edge_id_series: pd.Series, rm_value: int) -> pd.Series:
    """
    Convert edge_id series to int32-compatible values for GraphHopper.
    Assumes all ids start with '88', strips first two characters,
    then subtracts 3 from the left-most digit.
    """
    s = edge_id_series - rm_value
    return s.astype("int64")

In [8]:
route_network["edge_id"] = convert_edge_id_to_int32(route_network["edge_id"], 883000000000)

In [9]:
len(route_network["edge_id"].unique()), len(route_network["edge_id"]), max(route_network["edge_id"]), min(route_network["edge_id"])

(158167, 158167, 1152400188, 991900000)

In [10]:
route_network["track_segs"] = route_network["linestring"].str.replace(
    r"^LINESTRING\(|\)$", "", regex=True
)
route_network["track_segs"] = (
    route_network["track_segs"].str.replace(",", ";").replace(r"\s+", " ", regex=True)
)
route_network["track_segs"] = route_network["track_segs"].str.split(";")
route_network["track_segs"] = route_network["track_segs"].apply(
    lambda x: [tuple(float(coord) for coord in point.split()) for point in x]
)
route_network["track_segs"]

0         [(-122.732318937778, 47.8899192810059), (-122....
1         [(-122.71107852459, 47.8776508569717), (-122.7...
2         [(-122.707419991493, 47.8761515021324), (-122....
3         [(-122.707419991493, 47.8761515021324), (-122....
4         [(-122.715329825878, 47.8818699717522), (-122....
                                ...                        
158162    [(-121.777908504009, 47.4525502324104), (-121....
158163    [(-121.781320273876, 47.4532207846642), (-121....
158164    [(-121.784308254719, 47.4577805399895), (-121....
158165    [(-121.782489717007, 47.4503803253174), (-121....
158166    [(-121.754589378834, 47.4459412693977), (-121....
Name: track_segs, Length: 158167, dtype: object

In [11]:
route_network["speed"] = route_network["speed"] * 3.6  # Convert m/s to km/h
route_network["speed"]

0         80.0
1         40.0
2         40.0
3         40.0
4         40.0
          ... 
158162    40.0
158163    40.0
158164    40.0
158165    40.0
158166    60.0
Name: speed, Length: 158167, dtype: float64

In [12]:
type Coordinate = tuple[float, float]


def calculate_osm_node_id_map(coords: pd.Series) -> dict[Coordinate, int]:
    all_coords = set(coords.explode().unique().tolist())
    osm_node_map = {coord: i + 1 for i, coord in enumerate(all_coords)}
    return osm_node_map

In [13]:
osm_node_id_map = calculate_osm_node_id_map(route_network["track_segs"])
len(osm_node_id_map)

418444

In [14]:
def resolve_path_and_filename(path: Path, filename: str) -> Path:
    path = path.resolve().absolute()
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)
    path = path / filename if path.is_dir() else path

    return path


def generate_osm_xml_from_dataframe(
    df: pd.DataFrame,
    node_id_mapping: dict[Coordinate, int],
    output_directory: Path,
    output_filename: str = "newson_krumm_reconstructed.osm.xml",
) -> Path:
    current_timestamp = (
        datetime.now().isoformat(timespec="seconds") + "Z"
    )  # Formato ISO 8601 com 'Z'
    default_version = "1"
    default_changeset = "1"
    default_uid = "1"
    default_user = "reconstructed_map"

    # Iniciar a estrutura XML do OSM
    osm_root = ET.Element("osm", version="0.6", generator="CustomScriptFromNewsonKrumm")

    # 1. Adicionar todos os Nodes ao XML
    # Iteramos sobre o mapeamento que já tem os (lon, lat) únicos e seus IDs.
    # Lembre-se que a ordem é (lon, lat) no seu dataset e no node_id_mapping,
    # mas OSM exige (lat, lon) para os atributos `lat` e `lon` do elemento <node>.
    for (lon, lat), node_id in node_id_mapping.items():
        ET.SubElement(
            osm_root,
            "node",
            id=str(node_id),
            lat=str(lat),
            lon=str(lon),
            version=default_version,
            timestamp=current_timestamp,
            changeset=default_changeset,
            uid=default_uid,
            user=default_user,
        )

    # 2. Adicionar as Ways (arestas/ruas) ao XML
    for _, row in df.iterrows():
        edge_id = row["edge_id"]
        track_segments = row["track_segs"]
        two_way = row["two_way"]
        speed_ms = row["speed"]

        way_elem = ET.SubElement(
            osm_root,
            "way",
            id=str(edge_id),
            version=default_version,
            timestamp=current_timestamp,
            changeset=default_changeset,
            uid=default_uid,
            user=default_user,
        )

        for lon, lat in track_segments:
            node_osm_id = node_id_mapping[(lon, lat)]
            ET.SubElement(way_elem, "nd", ref=str(node_osm_id))

        ET.SubElement(way_elem, "tag", k="highway", v="unclassified")
        ET.SubElement(way_elem, "tag", k="maxspeed", v=str(round(speed_ms, 1)))
        ET.SubElement(way_elem, "tag", k="oneway", v="yes" if two_way == 0 else "no")

    tree = ET.ElementTree(osm_root)

    ET.indent(tree, space="  ", level=0)

    final_output_path = resolve_path_and_filename(output_directory, output_filename)

    tree.write(
        final_output_path,
        encoding="utf-8",
        xml_declaration=True,
    )

    print(f"OSM XML file generated and saved to {final_output_path}")
    return final_output_path

In [15]:
generate_osm_xml_from_dataframe(route_network, osm_node_id_map, DATA_BASE_PATH,output_filename="newson_krumm_reconstructed_2.osm.xml")

OSM XML file generated and saved to /home/jose_edsouza/Documentos/Faculdade/TCC/repo/dataset/newson-krumm/data/newson_krumm_reconstructed_2.osm.xml


PosixPath('/home/jose_edsouza/Documentos/Faculdade/TCC/repo/dataset/newson-krumm/data/newson_krumm_reconstructed_2.osm.xml')

In [16]:
def get_edge_ids_by_node_id(node_id: int, node_id_mapping: dict[Coordinate, int], df: pd.DataFrame) -> list:
    """
    Given a node_id, return a list of edge_ids (way IDs) that include this node.
    """
    # Find the coordinate corresponding to the node_id
    coord = next((c for c, nid in node_id_mapping.items() if nid == node_id), None)
    if coord is None:
        return []
    # Find all edge_ids where this coordinate appears in track_segs
    edge_ids = df[df["track_segs"].apply(lambda segs: coord in segs)]["edge_id"].tolist()
    return edge_ids

In [17]:
get_edge_ids_by_node_id(89470, osm_node_id_map, route_network)

[1130101679]